# Homework 1: Language Identification
11-411/11-611 Natural Language Processing (Fall 2026)

**Due:** Refer to Canvas and Gradescope for timeline.

**Submission:** Upload this single file named `HW1.ipynb` to Gradescope. Keep `train.tsv`, `test.tsv`, and `confusion_matrix.png` in the same directory while you work, but submit only the notebook unless your instructor says otherwise.

In this assignment, you will build a language identification classifier that distinguishes between six languages:

- [Hausa](https://en.wikipedia.org/wiki/Hausa_language)
- [Indonesian](https://en.wikipedia.org/wiki/Indonesian_language)
- [Manobo](https://en.wikipedia.org/wiki/Manobo_languages)
- [Nahuatl](https://en.wikipedia.org/wiki/Nahuatl)
- [Swahili](https://en.wikipedia.org/wiki/Swahili_language)
- [Tagalog](https://en.wikipedia.org/wiki/Tagalog_language)

All six are represented here using the [Latin script](https://en.wikipedia.org/wiki/Latin_script), with relatively few [diacritics](https://en.wikipedia.org/wiki/Diacritic), so simple script-based rules are not enough to distinguish them reliably.

A natural approach is to use character features. Individual character frequencies differ across languages, and short character sequences can capture useful orthographic and phonotactic patterns. Character $n$-grams therefore provide a compact feature representation for language identification. If $n$ becomes too large, however, the feature space becomes sparse and the model can overfit.

You will implement multinomial logistic regression (softmax regression) from scratch and use it to compare several character $n$-gram orders, inspect errors, and interpret learned feature weights.


## Requirements

You will:

1. Implement the gradient and training loop for multinomial logistic regression.
2. Implement inference for multinomial logistic regression.
3. Compare character $n$-gram orders $n \in \{1,2,3,4\}$ and select a model using macro-averaged F1.
4. Calculate and display a confusion matrix for a fixed trigram model.
5. Inspect a specific family of classification errors and formulate testable hypotheses.
6. Inspect feature weights and identify the most predictive character trigrams for each language.


## How to Use Jupyter Notebooks for This Assignment

This notebook mixes explanations, equations, implementation tasks, and short analyses. By the end of the assignment, you should be comfortable with character $n$-gram features, multinomial logistic regression, evaluation metrics, confusion matrices, and feature-weight interpretation.

### Types of cells
**Markdown cells** contain explanations, questions, equations, and response prompts.

**Code cells** contain provided code or places where you will implement a function or produce an output used by the grader.

Do not delete or reorder provided cells. You may use additional scratch cells at the end of the notebook while debugging, but your submitted notebook should run from top to bottom.

### Runtime volatility
If you restart your runtime or disconnect, variables and outputs in memory are lost. Save your notebook frequently and use **Run all** before submitting.

### Running on Google Colab
You may run the notebook in Google Colab. Upload `train.tsv`, `test.tsv`, and `confusion_matrix.png` into the same working directory as the notebook before running the data-loading cells.


## Important: Library Restrictions

**You must implement multinomial logistic regression from scratch using only the provided imports.**

You are **not allowed** to use:
- PyTorch, TensorFlow, JAX, or other deep-learning frameworks
- Scikit-learn or other machine-learning libraries
- External optimization libraries

You may use only the imported libraries below: `csv`, `random`, `numpy`, `numpy.typing`, `numpy.testing`, and `collections.Counter`.

## Imports
Do not change.


In [ ]:
import csv
import random

import numpy as np
import numpy.typing as npt
import numpy.testing as testing

from collections import Counter

# Revert to legacy printing style
np.set_printoptions(legacy='1.25')

## Helper Functions
### Understanding Data Structures

Throughout this assignment, you'll work with these key data structures:

1. **Raw observations**: `list[tuple[str, str]]` - List of (language_name, document_text) pairs
2. **Processed observations**: `list[tuple[np.ndarray, np.ndarray]]` - List of (one_hot_label, feature_vector) pairs
3. **Weight matrix W**: `np.ndarray` shape `[K, N]` where K=number of languages, N=number of features
   - **Important**: Column 0 contains bias terms, so remember to prepend 1 to feature vectors
4. **Feature map**: `dict[str, int]` - Maps n-gram strings to feature indices
5. **Language map**: `dict[str, np.ndarray]` - Maps language names to one-hot vectors

### Metrics for Binary Classification

In [ ]:
def precision(tp: int, fp: int) -> float:
    """Computes precision from true positives and false positives."""
    denom = tp + fp
    return tp / denom if denom else 0.0


def recall(tp: int, fn: int) -> float:
    """Computes recall from true positives and false negatives."""
    denom = tp + fn
    return tp / denom if denom else 0.0


def f_measure(beta: float, tp: int, fp: int, fn: int) -> float:
    """Computes F-beta from TP, FP, and FN."""
    p = precision(tp, fp)
    r = recall(tp, fn)
    denom = beta**2 * p + r
    return (1 + beta**2) * p * r / denom if denom else 0.0


def f1(tp: int, fp: int, fn: int) -> float:
    """Computes F1 from TP, FP, and FN."""
    return f_measure(1, tp, fp, fn)


### Micro-Averaged Metrics

In [ ]:
def micro_precision(tp: "dict[int, int]", fp: "dict[int, int]") -> float:
    """Computes micro-averaged precision."""
    tp_sum = sum(tp.values())
    fp_sum = sum(fp.values())
    denom = tp_sum + fp_sum
    return tp_sum / denom if denom else 0.0


def micro_recall(tp: "dict[int, int]", fn: "dict[int, int]") -> float:
    """Computes micro-averaged recall."""
    tp_sum = sum(tp.values())
    fn_sum = sum(fn.values())
    denom = tp_sum + fn_sum
    return tp_sum / denom if denom else 0.0


def micro_f1(tp: "dict[int, int]", fp: "dict[int, int]", fn: "dict[int, int]") -> float:
    """Computes micro-averaged F1."""
    mp = micro_precision(tp, fp)
    mr = micro_recall(tp, fn)
    return 2 * mp * mr / (mp + mr) if (mp + mr) else 0.0


### Macro-Averaged Metrics

In [ ]:
def _metric_classes(*counters) -> list[int]:
    """Return the sorted union of class IDs observed in metric counters."""
    return sorted(set().union(*(counter.keys() for counter in counters)))


def macro_precision(tp: "dict[int, int]", fp: "dict[int, int]") -> float:
    """Computes macro-averaged precision."""
    classes = _metric_classes(tp, fp)
    return np.mean([precision(tp[c], fp[c]) for c in classes]) if classes else 0.0


def macro_recall(tp: "dict[int, int]", fn: "dict[int, int]") -> float:
    """Computes macro-averaged recall."""
    classes = _metric_classes(tp, fn)
    return np.mean([recall(tp[c], fn[c]) for c in classes]) if classes else 0.0


def macro_f1(tp: "dict[int, int]", fp: "dict[int, int]", fn: "dict[int, int]") -> float:
    """Computes macro-F1 as the mean of the per-class F1 scores."""
    classes = _metric_classes(tp, fp, fn)
    return np.mean([f1(tp[c], fp[c], fn[c]) for c in classes]) if classes else 0.0


## Preprocess Data

### Loading data from files

We first need to load data from the provided TSV files. Each file is two columns, the language of the document and the document text, separated by a tab (`\t`) character. We load this data into a list of tuples, to maintain the coupling between each document and its label.

In [ ]:
def load_data(data_filename: str) -> list[tuple[str, str]]:
    """Load a two-column TSV as (language, document) tuples."""
    with open(data_filename, "r", errors="backslashreplace", encoding="utf-8") as f_in:
        observations = []
        for line in f_in:
            line = line.rstrip("\r\n")
            if not line:
                continue
            language, document = line.split("\t", 1)
            observations.append((language, document))
    return observations


### Feature Extraction

We will use ngrams as features, so we need to be able to extract them.

In [ ]:
def extract_ngrams(x: str, n=3) -> "list[str]":
    """Given a string, return all character ngrams of order `n`."""
    return ["".join(s) for s in (zip(*[x[i:] for i in range(n)]))]

### Language Codes to One-Hot Vectors
And we need a function to convert a language code into a **one-hot vector** (called $\mathbf{y}$).

In [ ]:
def to_onehot_vector(lang: str, langs: list[str]) -> np.ndarray:
    y = np.zeros(len(langs))
    y[langs.index(lang)] = 1
    return y

We need to be able to convert `dict`s of ngram counts to vectors of ngram counts (using a map from ngrams to dimensions of the vectors).

In [ ]:
def vectorize_ngrams(counter: dict[str, int], feature_map: dict[str, int]) -> np.ndarray:
    """
    Given a dict of ngram counts and a map from features to indices, returns a vector of ngram counts.

    Parameters
    ----------
    counter : dict
        Counter/dict of ngram counts
    feature_map : dict
        Map from ngrams to indices

    Returns
    -------
    np.ndarray
        A vector of ngram counts
    """
    feature_vector = np.zeros(len(feature_map))
    for ngram, count in counter.items():
        if ngram in feature_map:
            feature_vector[feature_map[ngram]] = count
    return feature_vector

And putting together the conversion from text into ngrams, and vectorizing the ngrams:

In [ ]:
def vectorize_document(document: str, feature_map: dict[str, int], ngram_length: int) -> np.ndarray:
    document_ngrams = extract_ngrams(document, ngram_length)
    vector = vectorize_ngrams(Counter(document_ngrams), feature_map)
    return vector

We need separate functions for preprocessing the training observations and the dev/test observations. The former function must return a map from language names to labels as one-hot vectors as well as a map from features (ngrams) to indices of vector dimensions.

In [ ]:
def preprocess_training_observations(
    training_observations: list[tuple[str, str]], n: int = 1
) -> tuple[list[tuple[np.ndarray, np.ndarray]], dict[str, np.ndarray], dict[str, int]]:
    langs = set()
    features = set()
    obs = []

    for lang, doc in training_observations:
        langs.add(lang)
        ngrams = extract_ngrams(doc, n)
        features = features | set(ngrams)
        obs.append((lang, ngrams))
    feature_map = {feature: idx for idx, feature in enumerate(sorted(features))}
    lang_list = list(sorted(langs))
    lang_map = {lang: to_onehot_vector(lang, lang_list) for lang in lang_list}

    obs = [
        (lang_map[lang], vectorize_ngrams(Counter(ngrams), feature_map)) for (lang, ngrams) in obs
    ]

    print(f"{len(obs)} training observations.")
    return obs, lang_map, feature_map

The function for preprocessing test observations (and dev observations) takes the feature map and the language map as arguments and returns only the list of labeled observations.

In [ ]:
def preprocess_test_observations(
    test_observations, feature_map: dict[str, int], lang_map: dict[str, np.ndarray], n: int = 1
) -> tuple[list[np.ndarray], list[np.ndarray]]:
    obs = []

    for i, (lang, doc) in enumerate(test_observations):
        vectorized_doc = vectorize_document(doc, feature_map, ngram_length=n)
        try:
            obs.append((lang_map[lang], vectorized_doc))
        except KeyError:
            print(f"Unkown language {lang} at index {i}. Known languages are: {lang_map.keys()}")
    print(f"{len(obs)} test observations.")
    return obs

### Classification Function

We also need to define the softmax function.

Softmax is technically

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum^K_{j=1} e^{z_j}}$$

However, if implemented naïvely, this is not numerically stable. Instead, we use:

$$\text{softmax}(z_i) = \frac{e^{z_i - \text{max}(z)}}{\sum^K_{j=1} e^{z_j - \text{max}(z)}}$$


In [ ]:
def softmax(z: npt.ArrayLike) -> npt.ArrayLike:
    """Compute the softmax of a vector `z`"""
    # exp(z) can get very large. For numerical stability, we subtract a vector of very large values (np.max(z)) from z.
    return np.exp(z - np.max(z)) / np.exp(z - np.max(z)).sum()

## Training the Classifier

### Compute the gradient

For one observation, the partial derivative of the negative log-likelihood (cross-entropy) loss is

$$\frac{\partial L_{CE}}{\partial \mathbf{w}_{k,i}}
= -(\mathbf{y}_k-\hat{\mathbf{y}}_k)\mathbf{x}_i$$

where $k$ indexes the **class** (rows of $\mathbf{W}$) and $i$ indexes the **feature** (columns of $\mathbf{W}$).

Implement `grad` so that it returns the complete $K \times N$ gradient matrix. The input `x` to this function already includes the bias feature when called from your training loop.

**This is the first piece of code that you will write.**


In [ ]:
def grad(W: np.ndarray, y: np.ndarray, x: np.ndarray) -> np.ndarray:
    """
    Calculate the gradient of the negative log-likelihood loss with respect to W.

    Parameters
    ----------
    W : np.ndarray
        Weight matrix of shape [K, N], where K is the number of classes.
    y : np.ndarray
        One-hot ground-truth label of shape [K].
    x : np.ndarray
        Feature vector of shape [N]. When used by the training loop, x includes
        the leading bias feature 1.

    Returns
    -------
    np.ndarray
        Gradient matrix with the same shape as W.
    """
    pass


### Training Loop

Iterate over the training observations for `epochs` passes. At the start of each epoch, visit the observations in a new random order. Use a non-mutating shuffle:

```python
shuffled_observations = random.sample(observations, len(observations))
```

For each observation:

1. prepend a bias feature of `1` to $\mathbf{x}$;
2. compute the class probabilities $\hat{\mathbf{y}} = \mathrm{softmax}(\mathbf{W}\mathbf{x})$ as part of your gradient calculation;
3. compute the gradient;
4. update the weights using learning rate $\eta$.

At the end of every epoch, evaluate the current model on the training observations and print at least micro-precision and micro-recall. We will not grade the exact formatting of these progress messages, but they are useful for checking that training is working.

### Bias term

If the original feature vector has length $N$, initialize $\mathbf{W}$ with shape $[K, N+1]$. Column 0 contains one bias weight per class. Therefore, whenever you multiply $\mathbf{W}$ by an observation vector, prepend `1` to the vector first.

The notebook sets the random seed immediately before each graded training run. Do **not** reset the random seed inside `train`.


In [ ]:
def train(observations: list[tuple[np.ndarray, np.ndarray]], eta: float, epochs: int = 1) -> np.ndarray:
    """
    Train a multinomial logistic-regression model with stochastic gradient descent.

    Parameters
    ----------
    observations : list[tuple[np.ndarray, np.ndarray]]
        Tuples of (y, x), where y is a one-hot label [K] and x is a feature
        vector [N].
    eta : float
        Learning rate.
    epochs : int
        Number of passes through the training data.

    Returns
    -------
    np.ndarray
        Trained weight matrix of shape [K, N+1], where column 0 contains biases.
    """
    pass


## Classification

The classification function is very simple.

In [ ]:
def classify(W: np.ndarray, x: np.ndarray) -> np.intp:
    """
    Return the index of the most probable language.

    Parameters
    ----------
    W : np.ndarray
        Weight matrix with one row per class and a bias in column 0.
    x : np.ndarray
        Real-valued feature vector without the bias feature.

    Returns
    -------
    np.intp
        Index of the predicted class.
    """
    pass


## Evaluate the Classifier

Then, a function to train and evaluate the classifier.

In [ ]:
def evaluate(train_set, test_set, eta, epochs=3):
    """Trains and evaluates a model."""
    print("Training model")
    W = train(train_set, eta, epochs=epochs)
    print("\nCLASSIFY")
    tp, fp, fn = Counter(), Counter(), Counter()
    for ref_lang_vec, x in test_set:
        ref_lang = np.argmax(ref_lang_vec)

        hyp_lang = classify(W, x)
        if hyp_lang == ref_lang:
            tp[ref_lang] += 1
        else:
            fp[hyp_lang] += 1
            fn[ref_lang] += 1
    # Print metrics

    test_macro_f1 = macro_f1(tp, fp, fn)
    test_micro_f1 = micro_f1(tp, fp, fn)
    print(f"macro-averaged F1:\t\t{test_macro_f1:.3f}")
    print(f"micro-averaged F1:\t\t{test_micro_f1:.3f}")
    return test_macro_f1, test_micro_f1

### Putting it all together

Load the data:

In [ ]:
train_observations = load_data("train.tsv")
test_observations = load_data("test.tsv")

### Inspecting the data

Before training, inspect the dataset and verify your assumptions.

For both the training and test sets, output a dictionary mapping each language label to its number of observations. Sort by **descending count**, breaking ties **alphabetically by label**.

Example format:

```python
{"hausa": 4000, "indonesian": 3500, ...}
```

Think about:
- How many training and test instances are there?
- Are the same labels present in both splits?
- Is the class distribution balanced?


In [ ]:
# this cell's output will be used for test 1.1
# your code here - train set

In [ ]:
# this cell's output will be used for test 1.2
# your code here - test set

### Set hyperparameters and parameters

Before training the model, we have to set the learning rate $\eta$ and the order of the ngrams used in feature extraction.

In [ ]:
eta = 0.0005  # Do not change this.
epochs = 4    # Do not change this.
ASSIGNMENT_SEED = 2028  # Do not change this.


In [ ]:
order_of_ngrams = 1  # Change this.

### Running our training and evaluation loops

Now train and evaluate your classifier using each character $n$-gram order in

$$n \in \{1,2,3,4\}.$$

Record the macro-F1 and micro-F1 for each value. Select the model with the **highest macro-F1**; use micro-F1 only as a tie-breaker. Before submission, set `order_of_ngrams` to your selected value and rerun the cells below.


In [ ]:
train_set, lang_map, feature_map = preprocess_training_observations(
    train_observations, n=order_of_ngrams
)
test_set = preprocess_test_observations(
    test_observations, feature_map, lang_map, n=order_of_ngrams
)

random.seed(ASSIGNMENT_SEED)
test_macro_f1, test_micro_f1 = evaluate(train_set, test_set, eta, epochs=epochs)


Print the tuple `(macro_f1, micro_f1)` for your **selected** $n$-gram order. Then explain:

1. Which value of $n$ did you select?
2. What did you observe across $n=1,2,3,4$?
3. Why might very small or larger values of $n$ perform differently on this dataset?

**Your response:** Replace this sentence with your explanation after running all four settings.


In [ ]:
# the output of this cell will be used for test 2.1
print((test_macro_f1, test_micro_f1))

## Inspecting Classification Results

Aggregate metrics tell us how well a classifier performs overall, but they do not explain *which* examples it gets wrong or why.

For the remainder of the assignment, use a newly trained **trigram** classifier trained for exactly **one epoch**. This fixed, intentionally under-trained model produces enough errors to support a useful error analysis while still performing reasonably well.


In [ ]:
# Re-train a fixed trigram classifier for inspection. Do not change these values.
INSPECTION_NGRAMS = 3
INSPECTION_EPOCHS = 1

train_set, lang_map, feature_map = preprocess_training_observations(
    train_observations, n=INSPECTION_NGRAMS
)
test_set = preprocess_test_observations(
    test_observations, feature_map, lang_map, n=INSPECTION_NGRAMS
)

random.seed(ASSIGNMENT_SEED)
W_inspect = train(train_set, eta, epochs=INSPECTION_EPOCHS)

tp, fp, fn = Counter(), Counter(), Counter()
for ref_lang_vec, x in test_set:
    ref_lang = np.argmax(ref_lang_vec)
    hyp_lang = classify(W_inspect, x)
    if hyp_lang == ref_lang:
        tp[ref_lang] += 1
    else:
        fp[hyp_lang] += 1
        fn[ref_lang] += 1

print(f"macro-averaged F1:\t\t{macro_f1(tp, fp, fn):.3f}")
print(f"micro-averaged F1:\t\t{micro_f1(tp, fp, fn):.3f}")


### Writing a prediction function

Write a function that takes a list of observations, and produces a list of either class indices, or class names based on a parameter. This will allow you both to look at individual results from evaluating on an existing set of data (like the test set), but also for you to evaluate your classifier on new data (i.e. any string). This will involve vectorizing the list of documents, and classifying those vectors. Once that's complete, construct an inverse language mapping, from predicted indices to the language they represent.

In [ ]:
def predict(
    documents: list[str],
    W: np.ndarray,
    feature_map: dict[str, int],
    lang_map: dict[str, np.ndarray],
    ngram_length: int,
    return_class_names=False,
):
    # your code here
    pass

Now, for each instance in the test set, print a tuple of the actual label, then the predicted label.

In [ ]:
# the output of this cell will be used for test 3.1
test_labels, test_documents = zip(*test_observations)
test_predictions = predict(
    test_documents,
    W_inspect,
    feature_map,
    lang_map,
    ngram_length=INSPECTION_NGRAMS,
    return_class_names=True,
)

print(list(zip(test_labels, test_predictions)))

### Plotting a Confusion Matrix

A *confusion matrix* is a $K \times K$ matrix in which cell $(i,j)$ counts the number of examples whose true class is $i$ and whose predicted class is $j$. Diagonal entries are correct classifications; off-diagonal entries are errors.

![A confusion matrix for a binary classification problem](confusion_matrix.png)

Write a function that returns a NumPy confusion matrix for the supplied labels and predictions. Use the class index encoded by `lang_map` for both rows and columns. Because `preprocess_training_observations` sorts language names before building the map, this also gives a deterministic class order.


In [ ]:
def get_confusion_matrix(
    labels: list[str], predictions: list[str], lang_map: dict[str, np.ndarray]
) -> np.ndarray:
    # your code here
    pass

Now, print your confusion matrix.

In [ ]:
# the output of this cell will be used for test 3.2
get_confusion_matrix(test_labels, test_predictions, lang_map)

From your confusion matrix, how many **Manobo** examples are misclassified as **Tagalog**? Print each of those misclassified documents on a separate line.


In [ ]:
# the output of this cell will be used for test 3.3
# your code here

The fixed inspection model should reveal a small set of Manobo examples that are predicted as Tagalog.

Formulate a hypothesis for why these examples are confused. Inspect the misclassified documents, the training data, and reliable resources about Manobo and Tagalog. Consider both:

- linguistic/data reasons (for example, related languages, shared strings, genre, names, or repeated constructions), and
- machine-learning reasons (feature overlap, class imbalance, sparse features, limited training, etc.).

Propose **2-3 concrete experiments** that could test your hypothesis.

**Your response:** Replace this sentence with your hypothesis and proposed experiments.


### Examining Feature Weights

The $K \times (N+1)$ weight matrix `W_inspect` can be interpreted one row at a time: each row contains the weights for one language class. Column 0 is the bias, while columns 1 through $N$ correspond to the $N$ entries in `feature_map`.

Construct an inverted feature map from vector index to trigram. Then extract the **Manobo feature-weight vector**, excluding the bias column, and print its shape.

**Important:** Use `W_inspect` from the fixed trigram inspection model above. If feature index `i` refers to an entry in `feature_map`, its weight is in column `i + 1` of `W_inspect`.


In [ ]:
# the output of this cell will be used for test 4.1
# Invert the feature map
inverted_feature_map = {v: k for k, v in feature_map.items()}

# your code here

Find the 10 features with the largest positive weights for **Manobo**. Print one tuple `(feature_name, feature_weight)` per line, from largest to smallest weight.

Example:

```text
("aaa", 0.04597442104739769)
("bbb", 0.03454984682736487)
```


In [ ]:
# the output of this cell will be used for test 4.2
# your code here

Now repeat the analysis for **every language**. For each language, print the language name and then a list of its 10 highest-weight trigrams, in descending weight order.

Example:

```text
hausa
[("aaa", 0.0459), ...]
indonesian
[("bbb", 0.0412), ...]
```

Remember to exclude the bias column from the feature ranking.


In [ ]:
# the output of this cell will be used for test 4.3
# your code here

Choose one language and consult its Wikipedia page or another reliable source. Do its top-10 trigrams make sense in light of what you learned about the language? Explain why or why not, considering both linguistic properties and how multinomial logistic regression learns weights from this particular dataset.

**Your response:** Replace this sentence with your analysis and cite the resource(s) you used.


### Extra Credit

Implement at least one experiment that tests your hypothesis about the Manobo-to-Tagalog errors. To receive credit, include:

- your hypothesis,
- the experiment and why it is an appropriate test,
- the result, and
- a short interpretation of what the result supports or fails to support.

Credit is based on the quality of the hypothesis, experimental design, evidence, and explanation rather than on whether the hypothesis turns out to be correct.


In [ ]:
# Implement extra credit here.